# Notebook 09 — Drug-Likeness Filters: Computational Rules for Oral Bioavailability and Safety

## Why This Matters

In the wet lab, you develop intuition for what makes a "good" drug candidate — solubility in the dosing vehicle, stability in plasma, permeability across Caco-2 monolayers. But when you're staring at 500,000 HTS hits, you can't run a Caco-2 assay on every one of them.

**Drug-likeness filters** encode decades of medicinal chemistry SAR knowledge as simple, computable rules. They translate wet-lab observations — "high-MW compounds tend to have poor oral absorption" — into numerical thresholds you can apply to an entire compound library in seconds.

**Key mindset shift:** These filters are *statistical tendencies*, not physical laws. They describe the property space where most orally bioavailable drugs live. Plenty of successful drugs violate one or more rules (cyclosporine, anyone?). The value is in **triage** — rapidly narrowing a massive hit list to a manageable set for medicinal chemistry follow-up, not in making absolute go/no-go decisions.

### What We'll Cover

| Filter | Year | Focus |
|--------|------|-------|
| Lipinski Ro5 | 1997 | Oral bioavailability (MW, LogP, HBD, HBA) |
| Veber | 2002 | Oral bioavailability (flexibility, polarity) |
| Ghose / Egan / Muegge | 1999–2002 | Complementary property windows |
| PAINS | 2010 | Assay interference (false positives) |
| BRENK / NIH / ZINC | Various | Structural alerts (toxicity, reactivity) |

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, FilterCatalog, Draw
import pandas as pd

## Embedded Dataset — A Mix of Real Drugs and Deliberate Filter Failures

We'll work with ~22 well-known drugs spanning different therapeutic areas and physicochemical profiles. The last two entries are deliberately chosen to *fail* certain filters, so we can see the filters in action rather than just watching everything pass.

In [ ]:
DRUGS = [
    ("Aspirin",        "CC(=O)OC1=CC=CC=C1C(=O)O"),
    ("Ibuprofen",      "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"),
    ("Caffeine",       "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"),
    ("Metformin",      "CN(C)C(=N)NC(=N)N"),
    ("Atorvastatin",   "CC(C)C1=C(C(=O)NC2=CC=CC=C2)C(=C(N1CCC(CC(CC(=O)O)O)O)C1=CC=C(F)C=C1)C1=CC=CC=C1"),
    ("Diazepam",       "CN1C(=O)CN=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C21"),
    ("Acetaminophen",  "CC(=O)NC1=CC=C(O)C=C1"),
    ("Naproxen",       "COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O"),
    ("Ciprofloxacin",  "OC(=O)C1=CN(C2CC2)C2=CC(N3CCNCC3)=C(F)C=C2C1=O"),
    ("Fluoxetine",     "CNCCC(OC1=CC=C(C(F)(F)F)C=C1)C1=CC=CC=C1"),
    ("Cetirizine",     "OC(=O)COCCN1CCN(CC1)C(C1=CC=CC=C1)C1=CC=C(Cl)C=C1"),
    ("Losartan",       "CCCCC1=NC(Cl)=C(N1CC1=CC=C(C=C1)C1=CC=CC=C1C1=NN=NN1)CO"),
    ("Sildenafil",     "CCCC1=NN(C)C2=C1NC(=NC2=O)C1=CC(=CC=C1OCC)S(=O)(=O)N1CCN(C)CC1"),
    ("Gabapentin",     "NCC1(CC(=O)O)CCCCC1"),
    ("Propranolol",    "CC(C)NCC(O)COC1=CC=CC2=CC=CC=C12"),
    ("Duloxetine",     "CNCC(OC1=CC=CC2=CC=CC=C12)C1=CC=CS1"),
    ("Metoprolol",     "COCCOCC(O)CNC(C)C"),
    ("Warfarin",       "CC(=O)CC(C1=CC=CC=C1)C1=C(O)C2=CC=CC=C2OC1=O"),
    ("Chloroquine",    "CCN(CC)CCCC(C)NC1=CC=NC2=CC(Cl)=CC=C12"),
    ("Allopurinol",    "O=C1N=CN=C2NNC=C12"),
    # --- Deliberate filter failures ---
    ("Rhodamine B (PAINS)",     "CCN(CC)C1=CC=C(C=C1)C(=C1C=CC(=[N+](CC)CC)C=C1)C1=CC=CC=C1C(=O)[O-]"),
    ("Cyclosporine A (MW>500)", "CC[C@@H]1NC(=O)[C@@H](CC(C)C)N(C)C(=O)[C@@H](CC(C)C)N(C)C(=O)[C@H](C)NC(=O)[C@H](C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](C)NC(=O)[C@H](C(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C1=O"),
]

df = pd.DataFrame(DRUGS, columns=["Name", "SMILES"])
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"])

print(f"Loaded {len(df)} molecules successfully.")
df[["Name", "SMILES"]].head(10)

## Lipinski's Rule of Five (Ro5) — The Granddaddy of Drug-Likeness Filters

### The Wet-Lab Backstory

In 1997, Chris Lipinski at Pfizer analyzed the physicochemical properties of ~2,245 drugs that had reached Phase II clinical trials. He noticed that orally active drugs clustered in a specific property window. The "Rule of Five" name comes from the fact that all cutoffs are multiples of 5:

| Property | Cutoff | Physical Rationale |
|----------|--------|--------------------|
| **MW** ≤ 500 | Larger molecules have lower passive membrane permeability (Fick's law — diffusion rate inversely proportional to molecular size) |
| **LogP** ≤ 5 | Too lipophilic → poor aqueous solubility → poor dissolution in GI fluid. Also correlates with plasma protein binding and CYP metabolism |
| **HBD** ≤ 5 | H-bond donors form strong interactions with water → desolvation penalty when crossing the lipid bilayer |
| **HBA** ≤ 10 | Same desolvation logic as HBD |

### Important Nuances

- **One violation is OK.** The original paper allows ≤ 1 violation. A compound with MW = 520 but perfect LogP/HBD/HBA still "passes."
- **Not for all drug classes.** Antibiotics, antifungals, vitamins, and natural products routinely violate Ro5 because they often use active transport rather than passive diffusion.
- **"Beyond Rule of 5" (bRo5)** is an active area — large molecules like cyclosporine achieve oral bioavailability through intramolecular H-bonds that mask polarity (chameleon effect).

In [ ]:
def lipinski_ro5(mol):
    """Check Lipinski Rule of Five. Returns (pass, violations dict).
    
    A molecule passes if it has at most 1 violation (per original paper).
    """
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)

    violations = {}
    if mw   > 500: violations["MW"]   = f"{mw:.0f} > 500"
    if logp > 5:   violations["LogP"] = f"{logp:.1f} > 5"
    if hbd  > 5:   violations["HBD"]  = f"{hbd} > 5"
    if hba  > 10:  violations["HBA"]  = f"{hba} > 10"

    return len(violations) <= 1, violations


# --- Apply to the dataset ---
print("Lipinski Rule of Five")
print("=" * 70)
for _, row in df.iterrows():
    passes, violations = lipinski_ro5(row["mol"])
    status = "PASS" if passes else "FAIL"
    viol_str = ", ".join(f"{k}: {v}" for k, v in violations.items()) or "none"
    print(f"  {row['Name']:30s} {status}  violations: {viol_str}")

## Veber Rules — Flexibility and Polarity

### The Wet-Lab Backstory

Veber et al. (2002, GSK) analyzed oral bioavailability in rats for 1,100 drug candidates and found two properties that were better predictors than Lipinski's:

| Property | Cutoff | Physical Rationale |
|----------|--------|--------------------|
| **Rotatable bonds** ≤ 10 | Flexible molecules pay a larger **conformational entropy penalty** upon binding to a target or crossing a membrane. Think of it this way: a floppy chain has to "freeze" into one conformation to thread through the lipid bilayer — that's thermodynamically expensive. |
| **TPSA** ≤ 140 A^2 | Topological Polar Surface Area sums the surface contributions of N, O, and attached H atoms. High TPSA = high polarity = poor passive diffusion across the nonpolar lipid bilayer interior. |

### Complementary to Lipinski

Veber captures information that Lipinski misses. A molecule can have MW = 400 and LogP = 3 (Lipinski-happy) but still fail orally if it has 15 rotatable bonds and TPSA = 180. **Use both filters together.**

In [ ]:
def veber_rules(mol):
    """Check Veber rules for oral bioavailability.
    
    Returns (pass, violations dict). All criteria must be met (no tolerance).
    """
    rot  = rdMolDescriptors.CalcNumRotatableBonds(mol)
    tpsa = Descriptors.TPSA(mol)

    violations = {}
    if rot  > 10:  violations["RotBonds"] = f"{rot} > 10"
    if tpsa > 140: violations["TPSA"]     = f"{tpsa:.0f} > 140"

    return len(violations) == 0, violations


# --- Apply to the dataset ---
print("Veber Rules (rotatable bonds + TPSA)")
print("=" * 70)
for _, row in df.iterrows():
    passes, violations = veber_rules(row["mol"])
    status = "PASS" if passes else "FAIL"
    viol_str = ", ".join(f"{k}: {v}" for k, v in violations.items()) or "none"
    print(f"  {row['Name']:30s} {status}  violations: {viol_str}")

## Ghose, Egan, and Muegge Filters — Complementary Property Windows

After Lipinski opened the door, several groups derived their own filters from different datasets and methodologies. Each captures slightly different aspects of drug-likeness:

### Ghose Filter (1999)
Derived from an analysis of the Comprehensive Medicinal Chemistry (CMC) database. Uses **molar refractivity (MR)** as a proxy for molecular polarizability/size, and total atom count instead of MW directly.

| Property | Range |
|----------|-------|
| 160 ≤ MW ≤ 480 | |
| -0.4 ≤ LogP ≤ 5.6 | |
| 40 ≤ Total atoms ≤ 130 | (including H) |
| 20 ≤ MR ≤ 130 | |

### Egan Filter (2000)
Focused purely on intestinal absorption using a simple 2D model with just two descriptors:

| Property | Cutoff |
|----------|--------|
| LogP ≤ 5.88 | |
| TPSA ≤ 131.6 | |

### Muegge Filter (2001)
The most comprehensive — derived from a "pharmacophore point filter" approach. Adds requirements for minimum molecular complexity (must have enough carbons and heteroatoms to be a plausible pharmacophore):

| Property | Range |
|----------|-------|
| 200 ≤ MW ≤ 600 | |
| -2 ≤ LogP ≤ 5 | |
| TPSA ≤ 150 | |
| Rings ≤ 7 | |
| Carbons > 4 | |
| Heteroatoms > 1 | |
| Rotatable bonds ≤ 15 | |
| HBD ≤ 5 | |
| HBA ≤ 10 | |

In [ ]:
def ghose_filter(mol):
    """Ghose filter (1999). Returns (pass, violations dict)."""
    mw    = Descriptors.MolWt(mol)
    logp  = Descriptors.MolLogP(mol)
    # Total atom count including implicit H
    n_atoms = mol.GetNumAtoms(onlyExplicit=False)
    mr    = Descriptors.MolMR(mol)

    violations = {}
    if not (160 <= mw <= 480):     violations["MW"]     = f"{mw:.0f} not in [160, 480]"
    if not (-0.4 <= logp <= 5.6):  violations["LogP"]   = f"{logp:.1f} not in [-0.4, 5.6]"
    if not (40 <= n_atoms <= 130): violations["Atoms"]  = f"{n_atoms} not in [40, 130]"
    if not (20 <= mr <= 130):      violations["MR"]     = f"{mr:.1f} not in [20, 130]"

    return len(violations) == 0, violations


def egan_filter(mol):
    """Egan filter (2000). Returns (pass, violations dict)."""
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)

    violations = {}
    if logp > 5.88:  violations["LogP"] = f"{logp:.1f} > 5.88"
    if tpsa > 131.6: violations["TPSA"] = f"{tpsa:.0f} > 131.6"

    return len(violations) == 0, violations


def muegge_filter(mol):
    """Muegge filter (2001). Returns (pass, violations dict)."""
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    rot  = rdMolDescriptors.CalcNumRotatableBonds(mol)

    # Count carbons and heteroatoms (non-C, non-H heavy atoms)
    n_carbons = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    n_hetero  = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1, 6))

    violations = {}
    if not (200 <= mw <= 600): violations["MW"]       = f"{mw:.0f} not in [200, 600]"
    if not (-2 <= logp <= 5):  violations["LogP"]     = f"{logp:.1f} not in [-2, 5]"
    if tpsa > 150:             violations["TPSA"]     = f"{tpsa:.0f} > 150"
    if rings > 7:              violations["Rings"]    = f"{rings} > 7"
    if n_carbons <= 4:         violations["Carbons"]  = f"{n_carbons} <= 4"
    if n_hetero <= 1:          violations["Hetero"]   = f"{n_hetero} <= 1"
    if rot > 15:               violations["RotBonds"] = f"{rot} > 15"
    if hbd > 5:                violations["HBD"]      = f"{hbd} > 5"
    if hba > 10:               violations["HBA"]      = f"{hba} > 10"

    return len(violations) == 0, violations


# --- Apply all three filters ---
for filter_name, filter_fn in [("Ghose", ghose_filter), ("Egan", egan_filter), ("Muegge", muegge_filter)]:
    print(f"\n{filter_name} Filter")
    print("=" * 70)
    pass_count = 0
    for _, row in df.iterrows():
        passes, violations = filter_fn(row["mol"])
        if passes:
            pass_count += 1
        status = "PASS" if passes else "FAIL"
        viol_str = ", ".join(f"{k}: {v}" for k, v in violations.items()) or "none"
        print(f"  {row['Name']:30s} {status}  violations: {viol_str}")
    print(f"  --- {pass_count}/{len(df)} passed ---")

## PAINS Filters — Catching Assay Artifacts Before They Waste Your Time

### The Wet-Lab Problem

If you've ever run an HTS campaign, you've seen *those* compounds — the ones that light up in every assay regardless of the target. You optimize them for months, only to discover they were never real hits. PAINS (Pan-Assay Interference Compounds) are structural motifs that give **false positives** through non-specific mechanisms:

| Mechanism | Example Motifs | What Happens in the Assay |
|-----------|---------------|---------------------------|
| **Redox cycling** | Quinones, catechols | Generate H2O2, which oxidizes assay reagents (especially in fluorescence-based readouts) |
| **Metal chelation** | Hydroxamic acids, 8-hydroxyquinolines | Chelate metal cofactors (Zn2+, etc.), inhibiting metalloenzymes nonspecifically |
| **Aggregation** | Highly lipophilic flat molecules | Form colloidal particles that sequester/denature enzymes |
| **Covalent modification** | Michael acceptors, rhodanines | React covalently with cysteine residues on any protein |

### Why Filter Computationally?

Running counter-screens (detergent assays for aggregation, jump-dilution for covalent) is possible but expensive at scale. PAINS filters (Baell & Holloway, 2010) encode 480 substructural patterns that flag these problematic motifs *before* you spend assay reagents.

**Caveat:** PAINS flags are not automatic rejections. Some PAINS motifs are present in approved drugs. The flag says "investigate further" — run orthogonal assays, test SAR around the flagged motif.

In [ ]:
# Build the PAINS filter catalog (RDKit ships with all 480 PAINS patterns)
pains_params = FilterCatalog.FilterCatalogParams()
pains_params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(pains_params)

print(f"PAINS catalog loaded: {pains_catalog.GetNumEntries()} substructure patterns")
print("=" * 70)

for _, row in df.iterrows():
    entry = pains_catalog.GetFirstMatch(row["mol"])
    if entry:
        print(f"  {row['Name']:30s} PAINS HIT: {entry.GetDescription()}")
    else:
        print(f"  {row['Name']:30s} clean")

## BRENK, NIH, and ZINC Structural Alert Filters

Beyond PAINS, RDKit ships with several additional structural alert catalogs. These flag **reactive**, **toxic**, or **unstable** functional groups:

| Catalog | Focus | Typical Alerts |
|---------|-------|----------------|
| **BRENK** | "Undesirable" substructures from Brenk et al. (2008) | Alkyl halides, Michael acceptors, sulfonyl halides, epoxides, acid anhydrides — groups that are chemically reactive and likely to cause off-target toxicity |
| **NIH** | NIH MLPCN (Molecular Libraries) filter | Reactive warheads, frequent hitters from PubChem screens, known assay-interfering motifs |
| **ZINC** | ZINC database "clean leads" filter | Reactive groups, toxic substructures, groups that complicate synthesis or are metabolically labile |

These are more aggressive than PAINS — they'll flag more compounds. Use them when building a screening library from scratch ("clean leads" curation) rather than for triaging existing HTS hits.

In [ ]:
# Combined structural alert filters — overview scan
print("Structural Alert Filters — Summary")
print("=" * 70)

for catalog_name, catalog_enum in [
    ("BRENK", FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK),
    ("NIH",   FilterCatalog.FilterCatalogParams.FilterCatalogs.NIH),
    ("ZINC",  FilterCatalog.FilterCatalogParams.FilterCatalogs.ZINC),
]:
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(catalog_enum)
    cat = FilterCatalog.FilterCatalog(params)

    flagged = []
    for _, row in df.iterrows():
        entry = cat.GetFirstMatch(row["mol"])
        if entry:
            flagged.append((row["Name"], entry.GetDescription()))

    print(f"\n  {catalog_name} ({cat.GetNumEntries()} patterns): "
          f"{len(flagged)}/{len(df)} molecules flagged")
    for name, desc in flagged:
        print(f"    - {name:30s} {desc}")

## Multi-Filter Pipeline — Putting It All Together

### The Real-World Workflow

In practice, you never apply just one filter. A typical computational triage after HTS looks like:

1. **Remove PAINS** — eliminate assay artifacts first (these aren't real hits)
2. **Apply Lipinski + Veber** — keep compounds in the oral bioavailability sweet spot
3. **Flag structural alerts** (BRENK) — highlight reactive/toxic groups for med chem review

The order matters: PAINS first (because no amount of property optimization fixes a false positive), then property filters, then structural alerts (which are softer flags).

Below, we build a unified pipeline and generate a summary DataFrame showing which filters each compound passes or fails.

In [ ]:
def drug_filter_pipeline(mol):
    """Apply all drug-likeness filters to a molecule.
    
    Returns a dict of boolean results for each filter category,
    plus an overall PASS_ALL flag.
    """
    results = {}

    # 1. Lipinski Ro5
    ro5_pass, _ = lipinski_ro5(mol)
    results["Lipinski"] = ro5_pass

    # 2. Veber rules
    veber_pass, _ = veber_rules(mol)
    results["Veber"] = veber_pass

    # 3. Ghose filter
    ghose_pass, _ = ghose_filter(mol)
    results["Ghose"] = ghose_pass

    # 4. Egan filter
    egan_pass, _ = egan_filter(mol)
    results["Egan"] = egan_pass

    # 5. Muegge filter
    muegge_pass, _ = muegge_filter(mol)
    results["Muegge"] = muegge_pass

    # 6. PAINS — clean means no hit (True = clean = good)
    results["PAINS_clean"] = pains_catalog.GetFirstMatch(mol) is None

    # 7. BRENK — clean means no hit
    brenk_params = FilterCatalog.FilterCatalogParams()
    brenk_params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    brenk_catalog = FilterCatalog.FilterCatalog(brenk_params)
    results["BRENK_clean"] = brenk_catalog.GetFirstMatch(mol) is None

    # Overall: must pass ALL filters
    results["PASS_ALL"] = all(results.values())

    return results


# --- Apply pipeline to all molecules ---
rows = []
for _, row in df.iterrows():
    result = drug_filter_pipeline(row["mol"])
    result["Name"] = row["Name"]
    rows.append(result)

df_results = pd.DataFrame(rows).set_index("Name")

# Display with a nice format: checkmarks for pass, X for fail
def fmt_bool(val):
    return "Pass" if val else "FAIL"

df_display = df_results.map(fmt_bool)
print(df_display.to_string())
print(f"\n{'=' * 70}")
print(f"Overall pass rate: {df_results['PASS_ALL'].sum()}/{len(df_results)} "
      f"({df_results['PASS_ALL'].sum()/len(df_results)*100:.0f}%)")

## Summary — When to Use Each Filter

| Filter | Best For | Strictness | Key Insight |
|--------|----------|------------|-------------|
| **Lipinski Ro5** | First-pass oral drug-likeness screen | Moderate (allows 1 violation) | The gold standard starting point; most widely cited |
| **Veber** | Complementing Lipinski with flexibility/polarity | Strict (no violations) | Catches floppy or polar molecules that Lipinski misses |
| **Ghose** | Tighter property window, includes MR | Moderate | Good for lead-like compounds; lower MW ceiling |
| **Egan** | Quick absorption check (2 descriptors) | Moderate | Simplest filter — only LogP and TPSA |
| **Muegge** | Most comprehensive property filter | Strict (many criteria) | Adds minimum complexity requirements (carbons, heteroatoms) |
| **PAINS** | Removing assay artifacts | Apply always | Not about drug-likeness — about data quality |
| **BRENK/NIH/ZINC** | Flagging reactive/toxic groups | Aggressive | Best for curating screening libraries from scratch |

### Practical Guidelines

1. **Always apply PAINS** — there's no reason to keep assay artifacts in your pipeline
2. **Lipinski + Veber** is a solid default combination for oral drug candidates
3. **Don't over-filter** — applying all filters simultaneously is too aggressive; you'll lose good compounds
4. **Context matters** — parenteral drugs don't need oral bioavailability filters; PROTACs and macrocycles live beyond Ro5
5. **Filters are for triage, not decisions** — a Lipinski violation doesn't kill a compound; it means you need more data

### Up Next

**Notebook 10** will explore molecular fingerprints and similarity searching — moving from rule-based filters to structure-based comparisons for finding analogs and SAR analysis.